# Notebook 03 - Stress Analysis and FEA Results

This is the first detailed solver notebook. It builds a piping model, loads or runs Code_Aster through the shared notebook helper, and only then displays stress, displacement, and reaction results.

You will do five things:

1. Build an operating load case for a restrained L-shaped pipe.
2. Load or run Code_Aster result artifacts.
3. Open the primary deformed-stress review view.
4. Inspect supplemental stress, displacement, and reaction views.


## 1. Environment Setup & Imports

In [ ]:
import sys
from pathlib import Path
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.plotting import plots

# Enable interactive notebook rendering
# Defaults to zoomable embedded HTML locally; set TUBA_NOTEBOOK_BACKEND=client or static to override.
from tuba.plotting.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 2. Building the Piping Model

We'll build a standard L-shaped piping layout with anchors at both ends and a lateral guide restraint close to the bend.

In [ ]:
model = Model("StressDemo")

# Material P265GH (allowable stress at different temperatures in Pa)
model.add_material(
    "P265GH",
    E=2.0e11,     # Young's modulus [Pa]
    nu=0.3,       # Poisson's ratio
    rho=7850.0,   # density [kg/m3]
    alpha=1.2e-5, # thermal expansion [1/K]
    allowable_stress={20.0: 137e6, 200.0: 120e6}
)

# Section: 4-inch Schedule 40
model.add_pipe_section("4inch_sch40", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)

# Routing the L-shape pipe
with model.pipe(section="4inch_sch40", material="P265GH") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(5.0)
    b.bend(radius=0.2, angle=90.0, plane="XY")
    b.run(3.0)
    b.end(support="anchor")

# Add a guide support at node N2 (the exit node of the bend)
model.add_support("N2", type="guide")

# Define operating load case
model.define_load_case("Operating", gravity=True, pressure=1.5e6, temperature=200.0)

print(model)

## 3. Load or Run Code_Aster Results

This notebook does not synthesize solver values. The next cell exports Code_Aster input files if needed, imports existing Code_Aster output tables from the same work directory, or runs the configured Code_Aster runtime when tables are missing.


In [ ]:
CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
# VS Code/Jupyter review defaults to committed real Code_Aster artifacts; set True only after the runtime doctor passes.
RUN_CODE_ASTER = False
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "stress_analysis_operating"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

if code_aster_run.ran_solver:
    print("Code_Aster solver executed for this notebook run.")
print(f"Loaded Code_Aster results from: {CODE_ASTER_WORK_DIR.resolve()}")
print(f"Node result count: {len(results.node_results)}")
print(f"Element result count: {len(results.element_results)}")

tuyau_subpoints = code_aster_artifact.result_state.metadata.get("tuyau_subpoints", [])
print(f"TUYAU sub-point rows: {len(tuyau_subpoints)}")
if tuyau_subpoints:
    first = tuyau_subpoints[0]
    print(f"First TUYAU sub-point position source: {first.get('position_source')}")
    print(f"First TUYAU sub-point shell display position: {first.get('display_position')}")

## 4. Interactive Result Review

Start with the combined deformed-stress view. The separate views below are useful when you need to isolate deformation, stress contours, displacement vectors, or reactions.


### A. Primary View: Combined Deformed Stress

This is the fastest engineering review view: warped geometry colored by Code_Aster stress output.


In [ ]:
plots.plot_deformed_stress(results, deform_scale=100.0, model=model, jupyter_backend=JUPYTER_BACKEND)

### B. Deformed Shape Only

Use this when displacement shape matters more than stress coloring.


In [ ]:
plots.plot_deformed(results, scale=100.0, model=model, jupyter_backend=JUPYTER_BACKEND)

### C. Stress Distribution Only

Use this when you want stress contours on the undeformed model.


In [ ]:
plots.plot_stress(results, model=model, jupyter_backend=JUPYTER_BACKEND)

### D. Displacement Vectors and Support Reactions

Use these views to inspect directional movement and reaction forces at restrained nodes.


In [ ]:
# Plot displacements as arrows
plots.plot_displacement_vectors(results, scale=100.0, model=model, jupyter_backend=JUPYTER_BACKEND)

# Plot support reactions
plots.plot_reactions(results, scale="auto", model=model, geometry_opacity=0.2, jupyter_backend=JUPYTER_BACKEND)

## 5. User-defined engineering checks

Tuba provides the Code_Aster results above. It does not implement piping-standard checks.
Select the applicable standard and edition, verify the model assumptions, and perform
the required calculations independently using these results. Finite-element Von Mises
stress alone does not establish piping-code compliance.

Next: `04_visualization_gallery.ipynb` exports the verified results for review.
